In [ ]:
import pandas as pd

%cd ..
from utils.estimators import DistributionUE, LabelProbability, SequenceProbability, DistributionOfProbUE, EmbeddingsBasedUE
from utils.metrics import ECE, Brier, ROCAUC
from utils.analysis_utils import *
from utils.statistical_tests import *
%cd ..

### Hyperparameters

In [ ]:
N = 210 
M = 8
K = 4  
S = 10  
top_p = 0.9
temperature = 0.6
replace_token = None
prompt_template = "default"
DATE = None
per = None
filter_by = {}
filter_samples = None

MODELS = ["medgemma-4b-it","qwen_2_vl_7b_instruct", "Qwen2.5-VL-7B-Instruct", "Molmo-7B-D-0924", 
    "llava_1.6_7b", "LLaVA-OneVision-1.5-8B-Instruct", 
    "medgemma-27b-it", "Qwen2.5-VL-32B-Instruct", "llava_1.6_34b", 
    "Molmo-72B-0924", "Qwen2.5-VL-72B-Instruct", "gpt-4.1-mini"]

metrics = [
    {
        "ue_method": LabelProbability(),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": SequenceProbability(aggregate="mean"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": SequenceProbability(aggregate="max"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionUE("SC", normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionUE("SE", normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": DistributionOfProbUE(normalize_options=False),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": EmbeddingsBasedUE(method_name="EE"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
    {
        "ue_method": EmbeddingsBasedUE(method_name="RDS"),
        "metrics": [ECE(normalize_min_max=True), Brier(normalize_min_max=True), ROCAUC()]
    },
]

os.makedirs("outputs/GMAI-MMBench/plots", exist_ok=True)

##  4.1 Main Results

In [ ]:

# Overwrite the aggregate_results_with_test in cde.utils.annotation_utils.py
def aggregate_results_with_test(stats_modified, stats_default, metrics: List[dict], model_name: str) -> dict:
    results = {}

    correctness_per_sample_modified = stats_modified["correctness_per_sample"]
    correctness_majority_voting_modified = stats_modified["correctness_majority_voting"]
    correctness_per_sample_default = stats_default["correctness_per_sample"]
    correctness_majority_voting_default = stats_default["correctness_majority_voting"]
    
    # n01: count where A incorrect, B correct
    # n10: count where A correct, B incorrect
    n01 = sum((correctness_per_sample_modified == 0) & (correctness_per_sample_default == 1))
    n10 = sum((correctness_per_sample_modified == 1) & (correctness_per_sample_default == 0))
    results["Accuracy"] = {
        "modified": float(correctness_per_sample_modified.mean()),
        "default": float(correctness_per_sample_default.mean()),
        "difference": float(correctness_per_sample_modified.mean() - correctness_per_sample_default.mean()),
        "p_value": mcnemar_from_counts(n01, n10)[1],
    }
    
    all_ue_estimates = {}
    for group in metrics:
        estimator = group["ue_method"]
        estimator_name = str(estimator)
        estimator_outputs_modified = estimator(stats_modified)
        estimator_outputs_default = estimator(stats_default)

        if len(estimator_outputs_modified) == len(correctness_per_sample_modified):
            target_modified = correctness_per_sample_modified
            target_default = correctness_per_sample_default
        elif len(estimator_outputs_modified) == len(correctness_majority_voting_modified):
            target_modified = correctness_majority_voting_modified
            target_default = correctness_majority_voting_default
        else:
            raise ValueError(f"Length mismatch between estimator outputs ({len(estimator_outputs_modified)}) and available targets ({len(correctness_per_sample_modified)} or {len(correctness_majority_voting_modified)}).")
        
        all_method_estimates = np.array([estimator_outputs_modified, estimator_outputs_default])
        if np.any(np.isnan(all_method_estimates)):
            # mask out samples where any method has NaN
            mask = ~np.isnan(all_method_estimates).any(axis=0)
            print(f"Mask shape: {mask.shape}, {mask.sum()} samples remain after filtering out NaNs across all methods.")
            
            # Apply mask
            estimator_outputs_default = estimator_outputs_default[mask]
            target_default = target_default[mask]
            estimator_outputs_modified = estimator_outputs_modified[mask]
            target_modified = target_modified[mask]

        min_max_norm = group["metrics"][0].normalize_min_max
        if min_max_norm:
            estimator_outputs_default = min_max_normalize(model_name, estimator_name, estimator_outputs_default)
            estimator_outputs_modified = min_max_normalize(model_name, estimator_name, estimator_outputs_modified)

        all_ue_estimates["default"] = {}
        all_ue_estimates["modified"] = {}
        all_ue_estimates["default"]["estimates"] = estimator_outputs_default.tolist()
        all_ue_estimates["default"]["targets"] = target_default.tolist()
        all_ue_estimates["modified"]["estimates"] = estimator_outputs_modified.tolist()
        all_ue_estimates["modified"]["targets"] = target_modified.tolist()

        if len(estimator_outputs_default) == 0:
            raise ValueError("default: All estimator values are NaN.")

        results["AUROC"] = compare_pairwise("auroc", all_ue_estimates)
        results["ECE"] = compare_pairwise("ece", all_ue_estimates)
        results["Brier"] = compare_pairwise("brier", all_ue_estimates)

        compared_avg_unc = compare_estimator_outputs(estimator_outputs_modified, estimator_outputs_default)
        results["Avg"] =  {
            "default": compared_avg_unc["avg_outputs_default"],
            "modified": compared_avg_unc["avg_outputs_modified"],
            "difference": compared_avg_unc["avg_outputs_modified"] - compared_avg_unc["avg_outputs_default"],
            "p_value": compared_avg_unc["p_value"],
        }

    return results

def get_stats(N, K, M, S, MODEL_NAME, PROMPT_TEMPLATE, temperature, top_p, replace_token=None, DATE=None,
                         metrics=[],
                         sample_K = None,
                         per: Union[str, None] = None,
                         filter_by: dict = {},
                         filter_samples: List[int] = []):
    """
    Load extracted and raw data, compute extraction and accuracy metrics,
    apply custom metrics (e.g., consistency, entropy), and return a single-row summary DataFrame.
    """
    ue_methods = [metric.get("ue_method") for metric in metrics if metric.get("ue_method") is not None]
    df_default = load_data(N, K, M, S, MODEL_NAME, PROMPT_TEMPLATE, temperature, top_p, 
        replace_token=None, DATE=DATE, filter_by=filter_by, filter_samples=filter_samples)
    df_plus_none = load_data(N, K, M, S, MODEL_NAME, PROMPT_TEMPLATE, temperature, top_p, 
        replace_token="plus_none", DATE=DATE, filter_by=filter_by, filter_samples=filter_samples)
    df_correct_to_none = load_data(N, K, M, S, MODEL_NAME, PROMPT_TEMPLATE, temperature, top_p, 
        replace_token="correct_to_none", DATE=DATE, filter_by=filter_by, filter_samples=filter_samples)

    if len(df_default) != len(df_plus_none) or len(df_default) != len(df_correct_to_none):
        # get the list of sample_idx, shuffle_idx, generation_idx triplets where each of the three dataframes have missing
        print("Lenghts are different, filtering")
        cols = ['sample_idx', 'shuffle_idx', 'generation_idx']
        common_samples = (
            pd.concat([
                df_default[cols],
                df_plus_none[cols],
                df_correct_to_none[cols],
            ])
            .value_counts()
            .reset_index(name='count')
        )
        common_samples = common_samples[common_samples['count'] == 3][cols]
        # remove these samples from all three dataframes
        cols = ['sample_idx', 'shuffle_idx', 'generation_idx']

        df_default = df_default.merge(common_samples, on=cols, how='inner')
        df_plus_none = df_plus_none.merge(common_samples, on=cols, how='inner')
        df_correct_to_none = df_correct_to_none.merge(common_samples, on=cols, how='inner')
        print(f"Len dfs: {len(df_default)}, {len(df_plus_none)}, {len(df_correct_to_none)} after removing samples with missing responses in any condition.")
    
    # Compute metrics per group
    S = None if K == 1 else S
    stats_default = calculate_sample_level_stats(df_default, S=S, load_embeddings=["embeddings-biomedical"])
    stats_plus_none = calculate_sample_level_stats(df_plus_none, S=S, load_embeddings=["embeddings-biomedical"])
    stats_correct_to_none = calculate_sample_level_stats(df_correct_to_none, S=S, load_embeddings=["embeddings-biomedical"])

    return stats_default, stats_plus_none, stats_correct_to_none

In [ ]:
significances = []
rows_topp = []
for model_name in MODELS:
    print(f"Processing {model_name}...")
    results_summaries, significance_list, _ = summarize_extraction(N, K, M, S, model_name, prompt_template, temperature, top_p, 
                            replace_token=None, DATE=DATE, metrics=metrics, sample_K=None, per=per, 
                            filter_by=filter_by, filter_samples=filter_samples, load_embeddings=["embeddings-biomedical"],
                            test=False, min_max="per_model")
    significances.extend(significance_list)
    rows_topp.append(results_summaries)

df_topp = pd.DataFrame([item[0] for item in rows_topp])
df_topp.columns

folder = "outputs/GMAI-MMBench/plots/4_1"
os.makedirs(folder, exist_ok=True)
df_topp.to_csv(f"{folder}/ue_results.csv", index=False)
significances = ["method,model,metric,in_best_group"] + significances

with open(f"{folder}/ue_results_significance_compare.csv", "a") as f:
    f.write("\n".join(significances))

## 4.2 Per category

In [ ]:
from collections import defaultdict

def build_tables_per_category(N, K, M, S, MODELS, PROMPT_TEMPLATES, temperature, top_p, replace_token, DATE, 
    metrics, per, filter_by: dict = {}, filter_samples: List[int] = [], folder="."):
    rows = []
    for model_name in MODELS:
        for prompt_template in PROMPT_TEMPLATES:
            print(f"Processing {model_name} with prompt {prompt_template}...")
            # try:
            summary_overall, significance_overall,  num_samples_overall = summarize_extraction(N, K, M, S, model_name, prompt_template, temperature, top_p, replace_token, DATE=DATE,
                                                    metrics=metrics, sample_K=None, per=None, filter_by=filter_by, filter_samples=filter_samples)
            summary_overall[0][per] = "Overall"
            significance_overall = [row.replace(model_name, "Overall") for row in significance_overall]
            summary, significance,  num_samples = summarize_extraction(N, K, M, S, model_name, prompt_template, temperature, top_p, replace_token, DATE=DATE,
                                                    metrics=metrics, sample_K=None, per=per, filter_by=filter_by, filter_samples=filter_samples)
            num_samples["Overall"] = num_samples_overall
            rows.append(summary+summary_overall)
            target_path = f"{folder}/{model_name}_per_{per}_significance.csv"
            if replace_token:
                target_path = f"{folder}/{model_name}_{replace_token}_per_{per}_significance.csv"
            if metrics:
                with open(target_path, "w") as f:
                    f.write(f"method,{per.replace('clinical VQA task', 'task')},metric,in_best_group\n")
                    f.write("\n".join(significance_overall + significance).replace("_embeddings-biomedical", ""))

    summaries = defaultdict(list)
    for results_model in rows:
        for results_category in results_model:
            category = results_category[per]
            summaries[category].append(results_category)

    categories = summaries.keys()

    if per == "modality":
        categories = ['CT', 'MRI', 'Endoscopy', 'Histopathology', 'Fundus Photography',  'X-ray', 'Microscopy', 'Dermoscopy', 'Overall']
    else:
        # order categories according to num_samples
        categories = sorted(categories, key=lambda x: num_samples[x], reverse=True)

    full_results = {}

    # First simple metrics like accuracy
    for metric in ["avg_accuracy_majority_voting", "avg_accuracy"]:
        rows_aggregate = []
        for model in MODELS:
            performance_values = []
            for category in categories:
                summaries_category = summaries[category]
                # find the item in summaries where item["MODEL_NAME"] == model
                for item in summaries_category:
                    if item["MODEL_NAME"] == model:
                        performance_values.append(item[metric])
                        break
            rows_aggregate.append(performance_values)

        rows_aggregate.append([num_samples[category] for category in categories])

        full_results[metric] = (pd.DataFrame(rows_aggregate, index=MODELS+["num_samples"], columns=list(categories)))

    # Then compound metrics like ECE + Semantic Entropy
    for metric in metrics:
        for ue_metric in metric["metrics"]:
            metric_name = f"{str(metric["ue_method"])}_{str(ue_metric)}"
            rows_aggregate = []
            for model in MODELS:
                performance_values = []
                for category in categories:
                    summaries_category = summaries[category]
                    # find the item in summaries where item["MODEL_NAME"] == model
                    for item in summaries_category:
                        if item["MODEL_NAME"] == model:
                            performance_values.append(item[metric_name])
                            break
                rows_aggregate.append(performance_values)

            rows_aggregate.append([num_samples[category] for category in categories])
            full_results[metric_name] = (pd.DataFrame(rows_aggregate, index=MODELS+["num_samples"], columns=list(categories)))

    return full_results

In [ ]:
MODELS = ["llava_1.6_34b"]
folder = "outputs/GMAI-MMBench/plots/histograms"
os.makedirs(folder, exist_ok=True)

per = "modality"
replace_token = None

df_accuracies = build_tables_per_category(N, K, M, S, MODELS, [prompt_template], temperature, top_p, 
                                          replace_token, DATE, metrics=metrics, per=per, filter_by=filter_by, 
                                          filter_samples=None, folder=folder)

df_accuracies['avg_accuracy_majority_voting'].to_csv(f"{folder}/avg_accuracy_majority_voting_per_{per}.csv")

# modalities = ["CT","MRI","Endoscopy","Histopathology","Fundus Photography","X-ray","Microscopy","Dermoscopy"]

other_category = df_accuracies["avg_accuracy_majority_voting"].columns.values.tolist()
if per == "department":
    mapping = {
            "Gastroenterology and Hepatology": "Gastro.",
            "Pulmonary Medicine": "Pulmonary",
            "Oncology (Medical)": "Oncology",
            "Orthopedic Surgery": "Orthopedic S.",
            "General Surgery": "General S.",
            "Hematology": "Hematology",
            "Urology": "Urology",
            "Laboratory Medicine and Pathology": "Pathology",
            "Cardiovascular Surgery": "Cardiovascular S.",
            "Ophthalmology": "Ophthalmology",
            "Dermatology": "Dermatology",
            "Otolaryngology (ENT)/Head and Neck Surgery": "Otolaryngology",
            "Nephrology and Hypertension": "Nephrology",
            "Obstetrics and Gynecology": "Obstetrics&Gynecology",
            "Neurosurgery": "Neurosurgery",
            "Sports Medicine": "Sports",
            "Endocrinology": "Endocrinology",
            "Infectious Diseases": "Infectious Dis."
        }
elif per == "clinical VQA task":
    mapping = {
        "Disease Diagnosis": "Diagnosis",
        "Surgical Instrument Recognition": "S. Instrument Rec.",
        "Cell Recognition": "Cell Rec.",
        "Image Quality Grading": "Img. Quality Grading",
        "Surgeon Action Recognition": "Action Rec.",
        "Organ Recognition - Thorax": "Thorax Rec.",
        "Blood Vessels Recognition": "Blood Vessels Rec.",
        "Bone": "Bone",
        "Organ Recognition - Head and Neck": "Head\&Neck Rec.",
        "Organ Recognition - Abdomen": "Abdomen Rec.",
        "Microorganism Recognition": "Microorganism Rec.",
        "Severity Grading": "Severity Grading",
        "Attribute Recognition": "Attribute Rec.",
        "Organ Recognition - Pelvic": "Pelvic Rec.",
        "Muscle": "Muscle",
        "Surgical Workflow Recognition": "S. Workflow Rec.",
        "Counting": "Counting"
    }
elif per == "perceptual granularity":
    mapping = {
        "Image level": "No annotation",
        "Contour level": "Contour",
        "Mask level": "Segmentation mask",
        "Box level": "Bounding box"
    }
else:
    mapping = {}

for model in MODELS:
    rows = []
    rows.append(["method"] + ["AUROC" for m in other_category] + ["ECE" for m in other_category] + ["Brier" for m in other_category])
    other_category_mapped = [(mapping.get(cat, cat)) for cat in other_category]
    rows.append(["method"] + other_category_mapped*3)
    for method in metrics:
        method_name = method["ue_method"].__str__()
        new_row = [method_name.replace("LabelNLL", "Label NLL").replace("MaxNLL", "Max NLL")]
        # for m in modalities:
        for m in other_category:
            new_row.append(float(df_accuracies[f"{method_name}_auroc"].loc[model, m]))
        for m in other_category:
            new_row.append(float(df_accuracies[f"{method_name}_ece"].loc[model, m]))
        for m in other_category:
            new_row.append(float(df_accuracies[f"{method_name}_brier"].loc[model, m]))
        rows.append(new_row)

    text = "\n".join([",".join(str(x) for x in row) for row in rows]).replace("_embeddings-biomedical", "")
    per = per.replace("clinical VQA task", "task")
    with open(f"{folder}/{model}_per_{per}.csv", "w") as f:
        f.write(text)


In [ ]:
estimator = "MeanNegativeLogLikelihood"
for per in ["modality", "department", "perceptual granularity", "clinical VQA task"]:
    ece = pd.read_csv(f"outputs/GMAI-MMBench/plots/{estimator}_ece_per_{per}.csv")
    brier = pd.read_csv(f"outputs/GMAI-MMBench/plots/{estimator}_brier_per_{per}.csv")
    roc_auc = pd.read_csv(f"outputs/GMAI-MMBench/plots/{estimator}_auroc_per_{per}.csv")
    num_samples = ece.iloc[-1]
    ece = ece.iloc[:-1]
    roc_auc = roc_auc.iloc[:-1]
    brier = brier.iloc[:-1]

    ece.rename(columns={"Unnamed: 0": "model"}, inplace=True)
    brier.rename(columns={"Unnamed: 0": "model"}, inplace=True)
    roc_auc.rename(columns={"Unnamed: 0": "model"}, inplace=True)

    merged = pd.merge(roc_auc, ece, on="model", suffixes=("_auroc", "_ece"))
    brier = brier.rename(columns={col: f"{col}_brier" for col in brier.columns[1:]})
    merged = pd.merge(merged, brier, on="model", suffixes=("", "_brier"))

    header_lines = [["model"], ["model"]]

    for category in merged.columns[1:]:
        task = category.split("_")[0]
        metric = category.split("_")[1]
        metric_map = {
            "auroc": "AUROC",
            "ece": "ECE",
            "brier": "Brier"
        }
        header_lines[0].append(metric_map[metric])
        header_lines[1].append(task)

    merged.columns = pd.MultiIndex.from_arrays(header_lines)

    merged.to_csv(f"{folder}/{estimator}_all_metrics_per_{per}.csv", index=False)

# 5.1 Compare performance under modifications to main results table in 4.1

In [ ]:
N = 210 
M = 8
K = 4  
S = 10  
top_p = 0.9
temperature = 0.6
replace_token = None
folder = "outputs/GMAI-MMBench/plots/deltas"
 
estimator_name = metrics[0]["ue_method"].__str__()
PROMPT_TEMPLATE = "default"
output_folder = f"{folder}/{estimator_name}"
os.makedirs(output_folder, exist_ok=True)
DATE = None
per = None
# filter_by = {"difficulty_label": "confusing"}
filter_by = {}
filter_samples = None
rows_topp = []

ue_methods = [metric.get("ue_method") for metric in metrics if metric.get("ue_method") is not None]
results_all_models = {}
for model_name in MODELS:
    stats_default, stats_plus_none, stats_correct_to_none = get_stats(N, K, M, S, model_name, PROMPT_TEMPLATE, temperature, top_p, replace_token=None, DATE=DATE,
            metrics=metrics)

    results_all_models[model_name] = {
        "plus_none": aggregate_results_with_test(stats_plus_none, stats_default, metrics, model_name=model_name),
        "correct_to_none": aggregate_results_with_test(stats_correct_to_none, stats_default, metrics, model_name=model_name),
    }

    with open(f"{output_folder}/{model_name}_results.json", "w") as f:
        json.dump(results_all_models[model_name], f, indent=4)

# Appendix E.3.2 Compare average baseline uncertainties across sample subgroups

In [4]:
import numpy as np

def compare_estimator_outputs_bootstrap(
    estimator_outputs,
    estimator_outputs_complement,
    alpha=0.05,
    n_bootstrap=10000,
    random_state=None,
):
    rng = np.random.default_rng(random_state)

    # Convert to numpy arrays and remove NaNs
    x = np.asarray([val for val in estimator_outputs if not np.isnan(val)], dtype=float)
    y = np.asarray([val for val in estimator_outputs_complement if not np.isnan(val)], dtype=float)

    if x.size < 2 or y.size < 2:
        raise ValueError("Both inputs must contain at least two values after removing NaNs.")

    avg_x = np.mean(x)
    avg_y = np.mean(y)

    # Relative delta average
    relative_delta_avg = (avg_x - avg_y) / avg_y if avg_y != 0 else np.nan

    # Bootstrap test: compute empirical distribution of mean difference
    combined = np.array(y)
    n = len(x)
    boot_diffs = np.empty(n_bootstrap)
    for i in range(n_bootstrap):
        sample = rng.choice(combined, size=n, replace=True)
        boot_diffs[i] = np.mean(sample) - avg_y

    # Two-sided p-value: how extreme is observed difference compared to bootstrap
    observed_diff = avg_x - avg_y
    p_value = np.mean(np.abs(boot_diffs) >= np.abs(observed_diff))

    return {
        "avg_outputs": avg_x,
        "avg_outputs_complement": avg_y,
        "relative_delta_avg": relative_delta_avg,
        "p_value": p_value,
        "significant": p_value < alpha,
    }

def compute_estimator_outputs(df, df_complement, metrics: List[dict]) -> dict:
    df = df[df["extracted_model_response_mapped"].notna()]
    df = df[df['extracted_model_response_mapped'] != "[]"]
    print(f"Len df after removing NaN and empty responses = {len(df)}")

    df_complement = df_complement[df_complement["extracted_model_response_mapped"].notna()]
    df_complement = df_complement[df_complement['extracted_model_response_mapped'] != "[]"]
    print(f"Len df_complement after removing NaN and empty responses = {len(df_complement)}")
    
    options = df['original_options'].tolist()
    options = [len(opt.keys()) for opt in options]
    stats = calculate_sample_level_stats(df, S=10)
    stats_complement = calculate_sample_level_stats(df_complement, S=10)

    result = {}
    for group in metrics:
        estimator = group["ue_method"]
        estimator_name = str(estimator)
        estimator_outputs = estimator(stats)
        print(f"Computing outputs for estimator: {estimator_name}, len: {len(estimator_outputs)}")
        estimator_outputs = np.asarray(estimator_outputs, dtype=float)
        estimator_outputs_complement = estimator(stats_complement)
        estimator_outputs_complement = np.asarray(estimator_outputs_complement, dtype=float)
        result[estimator_name] = compare_estimator_outputs_bootstrap(estimator_outputs, estimator_outputs_complement)

    return result

In [ ]:
model = "Qwen2.5-VL-32B-Instruct"
base_folder = "outputs/GMAI-MMBench"
ue_methods = [metric.get("ue_method") for metric in metrics if metric.get("ue_method") is not None]
def load_filtered_data(model_name, name):
    filter_df = pd.read_csv(f"{base_folder}/N_210-K_4-M_8/{model_name}-default-temp_0.6-topp_0.9-S_10/{name}.csv")
    return load_data_direct(model, f"{base_folder}/N_210-K_4-M_8/{model_name}-default-temp_0.6-topp_0.9-S_10/extracted.tsv", 
        replace_token=replace_token, filter=filter_df)

settings = ["plus_none_c_to_c", "plus_none_c_to_w", "plus_none_w_to_c", "plus_none_w_to_w", 
            "correct_to_none_c_to_c", "correct_to_none_c_to_w", "correct_to_none_w_to_c", "correct_to_none_w_to_w"]
all_dfs = {}
for setting in settings+["originally_correct", "originally_wrong"]:
    all_dfs[setting] = load_filtered_data(model, setting)
    print(f"Loaded data for setting {setting}, len: {len(all_dfs[setting])}")

nota_averages = {
    setting: compute_estimator_outputs(all_dfs[setting], all_dfs["originally_correct"].copy(), metrics) for setting in ["plus_none_c_to_c", "plus_none_c_to_w", "correct_to_none_c_to_c", "correct_to_none_c_to_w"]
}
for setting in ["plus_none_w_to_c", "plus_none_w_to_w","correct_to_none_w_to_c", "correct_to_none_w_to_w"]:
    nota_averages[setting] = compute_estimator_outputs(all_dfs[setting], all_dfs["originally_wrong"].copy(), metrics) 

for key, val in nota_averages.items():
    for estimator_name, stats in val.items():
        for metric_name, metric_value in stats.items():
            if type(metric_value) == np.bool_:
                nota_averages[key][estimator_name][metric_name] = bool(metric_value)
with open(f"/mnt/c/Users/P096811/OneDrive - Amsterdam UMC/Bureaublad/uncertainty/plots/5.2/{model}_nota_averages.json", "w") as f:
    json.dump(nota_averages, f, indent=4)

Loaded data for setting plus_none_c_to_c, len: 22130
Loaded data for setting plus_none_c_to_w, len: 6100
Loaded data for setting plus_none_w_to_c, len: 4160
Loaded data for setting plus_none_w_to_w, len: 31710
Loaded data for setting correct_to_none_c_to_c, len: 8710
Loaded data for setting correct_to_none_c_to_w, len: 19520
Loaded data for setting correct_to_none_w_to_c, len: 1940
Loaded data for setting correct_to_none_w_to_w, len: 33930
Loaded data for setting originally_correct, len: 28230
Loaded data for setting originally_wrong, len: 35870
Len df after removing NaN and empty responses = 22085
Len df_complement after removing NaN and empty responses = 28123
Computing outputs for estimator: LabelNLL, len: 2213


/mnt/c/Users/P096811/OneDrive - Amsterdam UMC/Bureaublad/uncertainty/code/Uncertainty_MedVQA_scripts/utils/estimators.py:237: RuntimeWarning: Mean of empty slice
  neg_label_log_likelihoods = [-np.nanmean(sample_nlls) for sample_nlls in label_log_likelihoods]


Computing outputs for estimator: ANLL, len: 2213
Computing outputs for estimator: MaxNLL, len: 2213
Computing outputs for estimator: SC, len: 2213
Computing outputs for estimator: SE, len: 2213
Computing outputs for estimator: PRO, len: 2213
Computing outputs for estimator: EE_embeddings-biomedical, len: 2213
Computing outputs for estimator: RDS_embeddings-biomedical, len: 2213
Len df after removing NaN and empty responses = 6038
Len df_complement after removing NaN and empty responses = 28123
Computing outputs for estimator: LabelNLL, len: 610
Computing outputs for estimator: ANLL, len: 610
Computing outputs for estimator: MaxNLL, len: 610
Computing outputs for estimator: SC, len: 610
Computing outputs for estimator: SE, len: 610
Computing outputs for estimator: PRO, len: 610
Computing outputs for estimator: EE_embeddings-biomedical, len: 610
Computing outputs for estimator: RDS_embeddings-biomedical, len: 610
Len df after removing NaN and empty responses = 8652
Len df_complement afte

In [ ]:
import numpy as np

BLUE_RGB   = np.array([0, 114, 178])
ORANGE_RGB = np.array([230, 159, 0])
WHITE_RGB  = np.array([255, 255, 255])

def format_relative_delta_cell(
    val,
    significant,
    precision=2,
):
    """
    Format relative delta for LaTeX using \\cellcolor[RGB]{...}.
    """
    if np.isnan(val):
        return "--"

    txt = f"{val:+.{precision}f}"

    if not significant:
        return txt

    # Normalize magnitude
    max_abs = 5.0 if val > 0 else 1.0
    t = min(abs(val) / max_abs, 1.0)
    t = t ** 0.5  if val > 0 else t ** 0.5  # Emphasize smaller values for negatives

    base = BLUE_RGB if val < 0 else ORANGE_RGB

    # Linear interpolation toward white
    rgb = (WHITE_RGB * (1 - t) + base * t).astype(int)

    return rf"\cellcolor[RGB]{{{rgb[0]},{rgb[1]},{rgb[2]}}} {txt}"


UE_methods = ["LabelNLL", "ANLL", "MaxNLL", "SC", "SE", "PRO", "EE_embeddings-biomedical", "RDS_embeddings-biomedical"]

# UE_methods = ["EE_embeddings-biomedical", "RDS_embeddings-biomedical"]
settings = ["plus_none_c_to_c", "plus_none_c_to_w", "plus_none_w_to_c", "plus_none_w_to_w", 
            "correct_to_none_c_to_c", "correct_to_none_c_to_w", "correct_to_none_w_to_c", "correct_to_none_w_to_w"]

for ue in UE_methods:
    row = [ue]

    # Example: two conditions × two transitions
    for key in settings:
        stats = nota_averages.get(key, {}).get(ue, {})
        val = stats.get("relative_delta_avg", np.nan)
        sig = stats.get("significant", False)

        row.append(format_relative_delta_cell(val, sig))

    print(" & ".join(row).replace("_embeddings-biomedical", "") + r" \\")

row = ["Num."]
for key in settings:
    row.append(str(len(all_dfs[key]["sample_idx"].unique())))
print(" & ".join(row) + r" \\")